In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
from time import strftime, sleep
from xmltodict import parse
from datetime import datetime, date
import re

In [2]:
def get_vacancies(query, area=40):
    url = "https://api.hh.kz/vacancies"
    all_items = []
    page = 0
    while True:
        params = {"per_page": 100, "text": query, "area": area, "page": page}
        data = requests.get(url, params=params).json()
        all_items.extend(data['items'])
        if page >= data['pages']-1:
            break
        page += 1
    return {'items': all_items}

In [3]:
queries = [
    "ML engineer", "Data analyst", "Data scientist", "Data science",
    "Аналитик данных", "дата аналитик", "мл инженер", "data engineer",
    "NLP engineer", "AI engineer", "machine learning engineer"
]

In [4]:
all_vacancies = []
seen_ids = set()

for query in queries:
    data = get_vacancies(query)
    for item in data['items']:
        if item['id'] not in seen_ids:
            all_vacancies.append(item)
            seen_ids.add(item['id'])

In [5]:
def get_details(vacancy):
    vacancy_id = vacancy['id']
    detail = requests.get(f"https://api.hh.kz/vacancies/{vacancy_id}").json()
    sleep(0.5)
    description_html = detail['description']
    clean_text = BeautifulSoup(description_html, "html.parser").get_text()
    return clean_text

def get_currency(title):
    date = strftime("%d.%m.%Y")
    url = f"https://nationalbank.kz/rss/get_rates.cfm?fdate={date}"
    content = requests.get(url).content
    parsed = parse(content)
    items = parsed['rates']['item']
    currency = next(item for item in items if item['title']==title)
    currency_rate = currency['description']
    return float(currency_rate)

def flatten_vacancy(v):
    sal_range = v.get('salary_range') or {}
    sal_old   = v.get('salary') or {}

    return {
        'id': v.get('id'),
        'salary_from': sal_range.get('from') or sal_old.get('from'),
        'salary_to':   sal_range.get('to')   or sal_old.get('to'),

        'area':         v.get('area', {}).get('name'),
        'experience':   v.get('experience', {}).get('id'),
        'employment':   v.get('employment', {}).get('id'),
        'schedule':     v.get('schedule', {}).get('id'),
        'work_format':  [f['id'] for f in v.get('work_format', [])],
        'professional_role': [r['name'] for r in v.get('professional_roles', [])],

        # Boolean
        'is_premium':       v.get('premium', False),
        'has_test':         v.get('has_test', False),
        'accept_temporary': v.get('accept_temporary', False),
        'is_gross':         sal_range.get('gross', False),
        'has_usd_in_description': False,

        'published_at': v.get('published_at'),

        'description': None,
        'name': v.get('name'),
    }

In [6]:
data = []
USD_RATE = get_currency('USD')
USD_PAT = re.compile(r'usd|\$|доллар|dollar', re.IGNORECASE)

for i in range(len(all_vacancies)):
    data.append(flatten_vacancy(all_vacancies[i]))
    description = get_details(all_vacancies[i])

    data[i]['description'] = description
    data[i]['has_usd_in_description'] = bool(USD_PAT.search(description or ''))
    
    sal = data[i]['salary_from'] or data[i]['salary_to']
    sal_obj = all_vacancies[i].get('salary_range') or all_vacancies[i].get('salary') or {}
    if sal_obj.get('currency') == 'USD':
        if data[i]['salary_from']: data[i]['salary_from'] *= USD_RATE
        if data[i]['salary_to']: data[i]['salary_to'] *= USD_RATE

    published_raw = data[i]['published_at']
    if published_raw:
        data[i]['published_at'] = datetime.fromisoformat(published_raw)

In [7]:
print(data[0].keys())
data[0]

dict_keys(['id', 'salary_from', 'salary_to', 'area', 'experience', 'employment', 'schedule', 'work_format', 'professional_role', 'is_premium', 'has_test', 'accept_temporary', 'is_gross', 'has_usd_in_description', 'published_at', 'description', 'name'])


{'id': '132031395',
 'salary_from': 700000,
 'salary_to': 1200000,
 'area': 'Алматы',
 'experience': 'between1And3',
 'employment': 'full',
 'schedule': 'fullDay',
 'work_format': ['ON_SITE', 'HYBRID'],
 'professional_role': ['Дата-сайентист'],
 'is_premium': False,
 'has_test': False,
 'accept_temporary': False,
 'is_gross': False,
 'has_usd_in_description': False,
 'published_at': datetime.datetime(2026, 4, 10, 16, 2, 39, tzinfo=datetime.timezone(datetime.timedelta(seconds=10800))),
 'description': 'Мы разрабатываем и развиваем продукт в сфере умного дома и безопасности. Проходим путь от разработки hardware решений до написание своих software решений. На данный момент усиляем команду в направлении сomputer vision: определение событий и контекста в видео потоке и отправка триггера на интерфейс. Обязанности:  Разработка и внедрение моделей компьютерного зрения для анализа видеопотока Распознавание действий (action recognition) и сценариев поведения Определение контекста происходящего (

In [8]:
df = pd.DataFrame(data)
df.head()

,id,salary_from,salary_to,area,experience,employment,schedule,work_format,professional_role,is_premium,has_test,accept_temporary,is_gross,has_usd_in_description,published_at,description,name
0,132031395,700000.0,1200000.0,Алматы,between1And3,full,fullDay,"[ON_SITE, HYBRID]",[Дата-сайентист],False,False,False,False,False,2026-04-10 16:02:39+03:00,Мы разрабатываем и развиваем продукт в сфере у...,ML Engineer / CV Engineer
1,131823118,9462.6,NaN,Астана,between1And3,full,remote,[REMOTE],[Специалист технической поддержки],False,False,True,False,True,2026-04-03 15:03:43+03:00,"Привет! Мы, ""Aspirity Solution"", студия разраб...",Customer Support Engineer
2,131990623,NaN,NaN,Алматы,between3And6,full,fullDay,[ON_SITE],[Дата-сайентист],False,False,False,False,False,2026-04-09 13:56:56+03:00,Обязанности: Обучать ML модели; Тестировать м...,ML-инженер/Data Scientist
3,131901669,NaN,NaN,Алматы,between3And6,full,fullDay,[ON_SITE],[Дата-сайентист],False,False,False,False,False,2026-04-07 10:18:52+03:00,Verigram is an international AI company focuse...,Middle ML engineer
4,130648164,400000.0,800000.0,Астана,between1And3,full,fullDay,[ON_SITE],[Дата-сайентист],False,False,False,False,False,2026-03-20 15:49:17+03:00,Обязанности: • Разработка и улучшение NLP/LLM ...,ML Engineer (NLP / RAG)


In [12]:
df.to_csv(f"data/raw/vacancies_{date.today()}.csv", index=False)